# Cohere Multimodal

## Install Library

In [ ]:
!pip install -U cohere hnswlib -q

In [ ]:
from google.colab import userdata
cohere_api = userdata.get('COHERE_API_KEY').strip()

In [ ]:
from PIL import Image
from IPython.display import display, HTML
import requests
import cohere
import base64
import hnswlib
import os

co = cohere.ClientV2(
    api_key=cohere_api
)  # Get your free API key: https://dashboard.cohere.com/api-keys

In [ ]:
!git clone https://github.com/cohere-ai/cohere-developer-experience.git

In [ ]:
cd cohere-developer-experience/notebooks/

In [ ]:
folder_path = "data/multimodal_semantic_search"
files = os.listdir(folder_path)

file_path = os.path.join(folder_path, files[1])
img = Image.open(file_path)
display(img)

In [ ]:
def image_to_base64_data_url(image_path):

    with open(image_path, "rb") as f:
        enc_img = base64.b64encode(f.read()).decode("utf-8")
        enc_img = f"data:image/jpeg;base64,{enc_img}"

    response = co.embed(
        model="embed-v4.0",
        images=[enc_img],
        input_type="image",
        embedding_types=["float"],
    )

    return response

In [ ]:
embeddings = []
file_paths = []

for file in files:
    file_path = os.path.join(folder_path, file)
    res = image_to_base64_data_url(file_path)
    file_paths.append(file_path)
    embeddings.append(res.embeddings.float[0])

In [ ]:
print(str(embeddings[0][:10])[:-1] + ", ...]")

In [ ]:
if embeddings:
    print(len(embeddings[0]))

In [ ]:
index = hnswlib.Index(space="cosine", dim=1536)
index.init_index(max_elements=len(embeddings), ef_construction=512, M=64)
index.add_items(embeddings, list(range(len(embeddings))))

In [ ]:
def retrieve_image(query):
    top_k = 5
    size = (200, 200)

    # Convert the query into embeddings
    query_emb = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
    ).embeddings.float

    # Retrieve the initial results from your vector db
    res = index.knn_query(query_emb, k=top_k)
    doc_index = res[0][0]
    doc_scores = res[1][0]

    # Let's view the response image:
    print(
        f"The most relevant photo in the top {top_k} retreived images with a distance score of: {doc_scores[0]:.2f}"
    )
    img = Image.open(file_paths[doc_index[0]])
    img_resized = img.resize(size)
    display(img_resized)

    # For the full list of images grabbed:
    print("-" * 100)
    print("All top k images:")
    for x in range(0, len(doc_index)):
        print(f"Ranking of Relevance:{x+1} with a distance of: {doc_scores[x]:.2f}")
        img = Image.open(file_paths[doc_index[x]])
        img_resized = img.resize(size)
        display(img_resized)

In [ ]:
# Set these paramters and query your database
query = "แมวตัวฟูสีเทา"

retrieve_image(query)

In [ ]:
def retrieve_image_top_k(query, top_k=2):
    size = (200, 200)

    # Convert the query into embeddings
    query_emb = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
    ).embeddings.float

    # Retrieve the initial results from your vector db
    res = index.knn_query(query_emb, k=top_k)
    doc_index = res[0][0]
    doc_scores = res[1][0]

    # For the full list of images grabbed:
    print("-" * 100)
    print("All top k images:")
    for x in range(0, len(doc_index)):
        print(f"Ranking of Relevance:{x+1} with a distance of: {doc_scores[x]:.2f}")
        img = Image.open(file_paths[doc_index[x]])
        img_resized = img.resize(size)
        display(img_resized)

In [ ]:
query = "รองเท้ายี่ห้อไนกี้"
retrieve_image_top_k(query)

In [ ]:
query = "กราฟการลงทุน"
retrieve_image_top_k(query)